## Vizualization MNIST DATASET / Segmenation Model

1. 단순한 MNIST 신경망
  - 입력층과 출력층으로만 구성
  - 입력층은 MNIST 이미지 크기인 784개의 뉴런을 가짐
  - 출력층은 숫자에 해당하는 10개의 뉴런을 가지며 소프트맥스 포함
  <img src = "images2/imagen01.png">

2. 단순한 MNIST 신경망의 가중치와 편향
  - 가중치 (행렬): 10 * 784개의 성분
  - 편향 (벡터): 10개의 성분
  <img src = "images2/imagen02.png">

3. 단순한 MNIST 신경망의 배치 구조
  - 단순한 구현을 위해 매스텝별 하나의 배치만 학습
  <img src = "images2/imagen03.png">

4. 병렬화 스킴
  - 데이터 병렬화
    <img src = "images2/imagen04.png">

  - 모델 병렬화 : 레이어와 피처
    <img src = "images2/imagen05.png">

  - 모델 병렬화 : 혼합
    <img src = "images2/imagen06.png">

5. 데이터 병렬화
  - 배치를 분할하여 각 코어 (랭크)에 할당
  - 가중치는 하나의 데이터 세트에 대해 계산되어 누적됨
  - 각 랭크에서 계산된 가중치를 누적
  <img src = "images2/imagen07.png">

  - Serial Code

In [ ]:
import numpy as np
import os

current_directory = os.getcwd()

def mnist_get_dataset(images_path, labels_path):
    """
    Description: get dataset from files
    INPUT: dataset paths
    OUTPUT:
        dataset: dict
            dataset.size
            dataset.label
            dataset.image
    """
    dataset = {}

    with open(os.path.join(current_directory, images_path), "rb") as mnist_file:
        dataset["images"] = np.frombuffer(
            mnist_file.read(), np.uint8, offset=16
        ).reshape(-1, 28 * 28)

    # Normalizate the data(images)
    dataset["images"] = dataset["images"] / 255.0

    with open(os.path.join(current_directory, labels_path), "rb") as mnist_file:
        dataset["labels"] = np.frombuffer(mnist_file.read(), np.uint8, offset=8)

    # (maybe) need one-hot encoding
    if len(dataset["images"]) != len(dataset["labels"]):
        print("Number of images does not match number of labels")
    else:
        dataset["size"] = len(dataset["images"])

    return dataset


# Fills the batch dataset with a subset of the parent dataset
def mnist_batch(dataset, size, number):
    batch = {}

    start_offset = int(size * number)
    if start_offset >= dataset["size"]:
        return 0

    batch["images"] = dataset["images"][start_offset : start_offset + size]
    batch["labels"] = dataset["labels"][start_offset : start_offset + size]
    batch["size"] = size

    if start_offset + batch["size"] > dataset["size"]:
        batch["size"] = dataset["size"] - start_offset
        print("batch_size:", batch["size"])

    return batch


In [ ]:
import numpy as np
import math
import pickle

MNIST_IMAGE_WIDTH = 28
MNIST_IMAGE_HEIGHT = 28
MNIST_IMAGE_SIZE = MNIST_IMAGE_WIDTH * MNIST_IMAGE_HEIGHT
MNIST_LABELS = 10


class NN:
    def __init__(self):
        self.b = np.zeros(MNIST_LABELS, dtype=np.float32)
        self.W = np.zeros((MNIST_LABELS, MNIST_IMAGE_SIZE), dtype=np.float32)

    def neural_network_random_weights(self):
        self.b = np.random.rand(MNIST_LABELS).astype(np.float32)
        self.W = np.random.rand(MNIST_LABELS, MNIST_IMAGE_SIZE).astype(np.float32)


class NN_Grad:
    def __init__(self):
        self.b_grad = np.zeros(MNIST_LABELS, dtype=np.float32)
        self.W_grad = np.zeros((MNIST_LABELS, MNIST_IMAGE_SIZE), dtype=np.float32)


# Calculate the softmax vector from the activations. This uses a more
# numerically stable algorithm that normalises the activations to prevent large exponents.
def neural_network_softmax(activations):
    e_activations = np.exp(activations - np.max(activations))
    return e_activations / np.sum(e_activations)


def neural_network_hypothesis(image, network):
    activations = np.dot(network.W, image) + network.b
    return neural_network_softmax(activations)


def neural_network_gradient_update(image, network: NN, gradient: NN_Grad, label):
    activations = neural_network_hypothesis(image, network)
    for i in range(MNIST_LABELS):
        b_grad = activations[i] - (1 if i == label else 0)
        gradient.b_grad[i] += b_grad
        gradient.W_grad[i] += b_grad * image
    return -math.log(activations[label])


def neural_network_training_step(dataset: dict, network: NN, learning_rate: float):
    total_loss = 0.0

    # Initialize gradient
    gradient = NN_Grad()

    # Calculate the gradient and the loss by looping through the training set
    for i in range(dataset["size"]):
        tmp = neural_network_gradient_update(
            dataset["images"][i], network, gradient, dataset["labels"][i]
        )
        total_loss += tmp

    # Apply gradient descent to the network
    for i in range(MNIST_LABELS):
        network.b[i] -= learning_rate * gradient.b_grad[i] / dataset["size"]
        network.W[i] -= learning_rate * gradient.W_grad[i] / dataset["size"]
    return total_loss


def save_network(network: NN, filename: str):
    with open(filename, "wb") as f:
        pickle.dump(network, f)


In [ ]:
# MNIST Classification
# Chanyoung Ahn (https://github.com/cold-young)
# 24.06.13

# Command Example:
# $ python main_mnist.py


import numpy as np
import matplotlib.pyplot as plt
#from mnist_file import mnist_get_dataset, mnist_batch
#from neural_network import (
#    NN,
#    NN_Grad,
#    neural_network_training_step,
#    neural_network_hypothesis,
#    MNIST_LABELS,
#    save_network,
#)

STEPS = 500
BATCH_SIZE = 400

# Dataset
# Downloaded from: http://yann.lecun.com/exdb/mnist/
data_sources = {
    "train_images": "data/train-images-idx3-ubyte",
    "train_labels": "data/train-labels-idx1-ubyte",
    "test_images": "data/t10k-images-idx3-ubyte",
    "test_labels": "data/t10k-labels-idx1-ubyte",
}


def calculate_accuracy(dataset: dict, network: NN):
    correct = 0
    for i in range(dataset["size"]):
        activations = neural_network_hypothesis(dataset["images"][i], network)
        predict = np.argmax(activations)
        if predict == dataset["labels"][i]:
            correct += 1

    return correct / dataset["size"]


def main():
    np.random.seed(0)
    network = NN()
    loss, accuracy = float, float

    # # Read the dataset from the ./data directory
    train_dataset = mnist_get_dataset(
        data_sources["train_images"], data_sources["train_labels"]
    )
    test_dataset = mnist_get_dataset(
        data_sources["test_images"], data_sources["test_labels"]
    )

    np.random.seed(0)

    network.neural_network_random_weights()
    batches = train_dataset["size"] / BATCH_SIZE

    for i in range(STEPS):
        # Initialize a new batch
        batch = mnist_batch(train_dataset, BATCH_SIZE, i % batches)

        # Run one step of gradient descent and calculate the loss
        loss = neural_network_training_step(batch, network, 0.05)
        accuracy = calculate_accuracy(test_dataset, network)
        size = batch["size"]
        result = "Step: {0:3}  Average Loss: {1:2.3f} \t Accuracy: {2:3.3f}".format(
            i, loss / size, accuracy
        )
        print(result)
    save_network(network, "./serial_model.pkl")


if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        import traceback

        traceback.print_exc()


  - Verification

In [ ]:
import pickle

your_model_name='serial_model.pkl'

def load_network(filename):
    with open(filename, 'rb') as f:
        network = pickle.load(f)
    return network

def calculate_accuracy(dataset: dict, network: NN):
    correct = 0

    for i in range(dataset["size"]):
        activations = neural_network_hypothesis(dataset["images"][i], network)
        predict = np.argmax(activations)
        if predict == dataset["labels"][i]:
            correct += 1

    return correct / dataset["size"]

In [ ]:
# Load Test dataset
data_sources = {
    "train_images": "data/train-images-idx3-ubyte",
    "train_labels": "data/train-labels-idx1-ubyte",
    "test_images": "data/t10k-images-idx3-ubyte",
    "test_labels": "data/t10k-labels-idx1-ubyte",
}
test_dataset = mnist_get_dataset(
    data_sources["test_images"], data_sources["test_labels"]
)

In [ ]:
# Visualize Test dataset
num_examples = 5
seed = 147197952744
rng = np.random.default_rng(seed)

fig, axes = plt.subplots(1, num_examples)
for sample, ax in zip(rng.choice(test_dataset["images"], size=num_examples, replace=False), axes):
    ax.imshow(sample.reshape(28, 28), cmap='gray')

In [ ]:
# Load your network model
network = load_network(your_model_name)

# Test accuarcy from test_dataset
accuracy = calculate_accuracy(test_dataset, network)
result = "Total Accuracy: {0:3.3f} %".format(accuracy*100)
print(result)

 - Parallel Code

In [ ]:
%%writefile examples/mnist_file.py

import numpy as np
import os

current_directory = os.getcwd()


def mnist_get_dataset(images_path, labels_path):
    """
    Description: get dataset from files
    INPUT: dataset paths
    OUTPUT:
        dataset: dict
            dataset.size
            dataset.label
            dataset.image
    """
    dataset = {}

    with open(os.path.join(current_directory, images_path), "rb") as mnist_file:
        dataset["images"] = np.frombuffer(
            mnist_file.read(), np.uint8, offset=16
        ).reshape(-1, 28 * 28)

    # Normalizate the data(images)
    dataset["images"] = dataset["images"] / 255.0

    with open(os.path.join(current_directory, labels_path), "rb") as mnist_file:
        dataset["labels"] = np.frombuffer(mnist_file.read(), np.uint8, offset=8)

    # (maybe) need one-hot encoding
    if len(dataset["images"]) != len(dataset["labels"]):
        print("Number of images does not match number of labels")
    else:
        dataset["size"] = len(dataset["images"])

    return dataset


# Fills the batch dataset with a subset of the parent dataset
def mnist_batch(dataset, total_size, number):
    """
    total_size = total size of batch (BATCH_SIZE)
    size = size of each processor (iend1 - ista1 +1)
    number = i % batches
    """
    batch = {}

    start_offset = int(total_size * number)
    if start_offset >= dataset["size"]:
        return 0

    batch["images"] = dataset["images"][start_offset : start_offset + total_size]
    batch["labels"] = dataset["labels"][start_offset : start_offset + total_size]
    batch["size"] = total_size

    if start_offset + batch["size"] > dataset["size"]:
        batch["size"] = dataset["size"] - start_offset
        print("batch_size:", batch["size"])

    return batch


In [ ]:
%%writefile examples/neural_network.py

import numpy as np
from mpi4py import MPI
import math
import pickle

MNIST_IMAGE_WIDTH = 28
MNIST_IMAGE_HEIGHT = 28
MNIST_IMAGE_SIZE = MNIST_IMAGE_WIDTH * MNIST_IMAGE_HEIGHT
MNIST_LABELS = 10


class NN:
    def __init__(self):
        self.b = np.zeros(MNIST_LABELS, dtype=np.float32)
        self.W = np.zeros((MNIST_LABELS, MNIST_IMAGE_SIZE), dtype=np.float32)

    def neural_network_random_weights(self):
        self.b = np.random.rand(MNIST_LABELS).astype(np.float32)
        self.W = np.random.rand(MNIST_LABELS, MNIST_IMAGE_SIZE).astype(np.float32)


class NN_Grad:
    def __init__(self):
        self.b_grad = np.zeros(MNIST_LABELS, dtype=np.float32)
        self.W_grad = np.zeros((MNIST_LABELS, MNIST_IMAGE_SIZE), dtype=np.float32)


# Calculate the softmax vector from the activations. This uses a more
# numerically stable algorithm that normalises the activations to prevent large exponents.
def neural_network_softmax(activations):
    e_activations = np.exp(activations - np.max(activations))
    return e_activations / np.sum(e_activations)


def neural_network_hypothesis(image, network):
    activations = np.dot(network.W, image) + network.b
    return neural_network_softmax(activations)


def neural_network_gradient_update(image, network: NN, gradient: NN_Grad, label):
    activations = neural_network_hypothesis(image, network)
    for i in range(MNIST_LABELS):
        b_grad = activations[i] - (1 if i == label else 0)
        gradient.b_grad[i] += b_grad
        gradient.W_grad[i] += b_grad * image
    return -math.log(activations[label])


def neural_network_training_step(
    dataset: dict,
    network: NN,
    learning_rate: float,
    ista: int,
    iend: int,
    total_size: int,
):
    local_loss = np.array(0.0, dtype=np.float32)
    total_loss = np.array(0.0, dtype=np.float32)

    # Initialize gradient
    gradient = NN_Grad()

    # Calculate the gradient and the loss by looping through the training set
    for i in range(ista, iend + 1):
        local_loss += neural_network_gradient_update(
            dataset["images"][i], network, gradient, dataset["labels"][i]
        )

    comm = MPI.COMM_WORLD
    comm.Allreduce([local_loss, MPI.FLOAT], [total_loss, MPI.FLOAT], op=MPI.SUM)
    comm.Allreduce(MPI.IN_PLACE, gradient.W_grad, op=MPI.SUM)
    comm.Allreduce(MPI.IN_PLACE, gradient.b_grad, op=MPI.SUM)

    # Apply gradient descent to the network
    for i in range(MNIST_LABELS):
        network.b[i] -= learning_rate * gradient.b_grad[i] / float(total_size)
        network.W[i] -= learning_rate * gradient.W_grad[i] / float(total_size)
    return total_loss


def save_network(network: NN, filename: str):
    with open(filename, "wb") as f:
        pickle.dump(network, f)



In [ ]:
%%writefile examples/main_mnist_parallel.py

# MNIST Classification
# Chanyoung Ahn (https://github.com/cold-young)
# 24.06.13

# Command Example:
# $ mpirun -np 4 python main_mnist.py

import numpy as np
import matplotlib.pyplot as plt
from mnist_file import mnist_get_dataset, mnist_batch
from neural_network import (
    NN,
    NN_Grad,
    neural_network_training_step,
    neural_network_hypothesis,
    save_network,
)
from mpi4py import MPI

STEPS = 500
BATCH_SIZE = 400

# Dataset
# Downloaded from: http://yann.lecun.com/exdb/mnist/
data_sources = {
    "train_images": "../data/train-images-idx3-ubyte",
    "train_labels": "../data/train-labels-idx1-ubyte",
    "test_images": "../data/t10k-images-idx3-ubyte",
    "test_labels": "../data/t10k-labels-idx1-ubyte",
}


def calculate_accuracy(dataset: dict, network: NN, ista: int, iend: int):
    correct = np.array(0.0, dtype=np.float32)
    total_correct = np.array(0.0, dtype=np.float32)
    for i in range(ista, iend + 1):
        activations = neural_network_hypothesis(dataset["images"][i], network)
        predict = np.argmax(activations)
        if predict == dataset["labels"][i]:
            correct += 1.0

    comm = MPI.COMM_WORLD
    comm.Allreduce([correct, MPI.FLOAT], [total_correct, MPI.FLOAT], op=MPI.SUM)
    return total_correct / dataset["size"]


def para_range(N: int, nproc: int, myrank: int):
    iwork1 = N // nproc
    iwork2 = N % nproc
    ista = myrank * iwork1 + min(myrank, iwork2)
    iend = ista + iwork1 - 1
    if iwork2 > myrank:
        iend += 1
    return ista, iend


def main():
    np.random.seed(0)
    network = NN()
    loss, accuracy = float, float

    # MPI Initialize
    comm = MPI.COMM_WORLD
    myrank = comm.Get_rank()  # current_process
    nproc = comm.Get_size()  # np

    # Read the dataset from the ./data directory
    train_dataset = mnist_get_dataset(
        data_sources["train_images"], data_sources["train_labels"]
    )
    test_dataset = mnist_get_dataset(
        data_sources["test_images"], data_sources["test_labels"]
    )

    # Calculate how many batches (so we know when to wrap around)
    batches = train_dataset["size"] / BATCH_SIZE

    ista1, iend1 = para_range(BATCH_SIZE, nproc, myrank)
    ista2, iend2 = para_range(test_dataset["size"], nproc, myrank)

    network.neural_network_random_weights()

    for i in range(STEPS):
        # Initialize a new batch
        batch = mnist_batch(train_dataset, BATCH_SIZE, i % batches)
        loss = neural_network_training_step(
            batch, network, 0.05, ista1, iend1, BATCH_SIZE
        )
        accuracy = calculate_accuracy(test_dataset, network, ista2, iend2)

        if myrank == 0:
            result = "Step: {0:3}  Average Loss: {1:2.3f} \t Accuracy: {2:3.3f}".format(
                i, loss / BATCH_SIZE, accuracy
            )
            print(result)

    if myrank == 0:
        save_network(network, "./parallel_model.pkl")


if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        import traceback

        traceback.print_exc()


In [ ]:
! mpiexec -np 2 python examples/main_mnist_parallel.py

6. 데이터 병렬화 : 은닉층 추가
  - 가중치 행렬과 편향 벡터 세트 추가
  - 데이터 병렬화 스킴은 동일
  <img src = "images2/imagen08.png">

 - Serial code

In [ ]:
import numpy as np
import os

current_directory = os.getcwd()


def mnist_get_dataset(images_path, labels_path):
    """
    Description:
    INPUT:
    OUTPUT:

    dataset.size
    dataset.images
    dataset.labels
    """
    dataset = {}

    with open(os.path.join(current_directory, images_path), "rb") as mnist_file:
        dataset["images"] = np.frombuffer(
            mnist_file.read(), np.uint8, offset=16
        ).reshape(-1, 28 * 28)

    # Normalizate the data(images)
    dataset["images"] = dataset["images"] / 255.0

    with open(os.path.join(current_directory, labels_path), "rb") as mnist_file:
        dataset["labels"] = np.frombuffer(mnist_file.read(), np.uint8, offset=8)

    # (maybe) need one-hot encoding
    if len(dataset["images"]) != len(dataset["labels"]):
        print("Number of images does not match number of labels")
    else:
        dataset["size"] = len(dataset["images"])

    return dataset


# Fills the batch dataset with a subset of the parent dataset
def mnist_batch(dataset, size, number):
    """
    Description:
    INPUT:
    OUTPUT:
    """
    batch = {}

    start_offset = int(size * number)
    if start_offset >= dataset["size"]:
        return 0

    batch["images"] = dataset["images"][start_offset : start_offset + size]
    batch["labels"] = dataset["labels"][start_offset : start_offset + size]
    batch["size"] = size

    # print(f"start_offset:{start_offset}, batch:", batch["size"], ", dataset_size:", dataset["size"])
    if start_offset + batch["size"] > dataset["size"]:
        batch["size"] = dataset["size"] - start_offset
        print("batch_size:", batch["size"])

    return batch


In [ ]:
import numpy as np
import math
import pickle

MNIST_IMAGE_WIDTH = 28
MNIST_IMAGE_HEIGHT = 28
MNIST_IMAGE_SIZE = MNIST_IMAGE_WIDTH * MNIST_IMAGE_HEIGHT
MNIST_LABELS = 10
HIDDEN_SIZE = 128


class NN:
    def __init__(self):
        self.b1 = np.zeros(HIDDEN_SIZE, dtype=np.float32)
        self.W1 = np.zeros((HIDDEN_SIZE, MNIST_IMAGE_SIZE), dtype=np.float32)
        self.b2 = np.zeros(MNIST_LABELS, dtype=np.float32)
        self.W2 = np.zeros((MNIST_LABELS, HIDDEN_SIZE), dtype=np.float32)

    def neural_network_random_weights(self):
        self.b1 = np.random.rand(HIDDEN_SIZE).astype(np.float32)
        self.W1 = np.random.rand(HIDDEN_SIZE, MNIST_IMAGE_SIZE).astype(np.float32)
        self.b2 = np.random.rand(MNIST_LABELS).astype(np.float32)
        self.W2 = np.random.rand(MNIST_LABELS, HIDDEN_SIZE).astype(np.float32)


class NN_Grad:
    def __init__(self):
        self.b1_grad = np.zeros(HIDDEN_SIZE, dtype=np.float32)
        self.W1_grad = np.zeros((HIDDEN_SIZE, MNIST_IMAGE_SIZE), dtype=np.float32)
        self.b2_grad = np.zeros(MNIST_LABELS, dtype=np.float32)
        self.W2_grad = np.zeros((MNIST_LABELS, HIDDEN_SIZE), dtype=np.float32)


def neural_network_softmax(activations):
    e_activations = np.exp(activations - np.max(activations))
    return e_activations / np.sum(e_activations)


def neural_network_hypothesis(image, network):
    hidden_activations = np.maximum(
        0, np.dot(network.W1, image) + network.b1
    )  # ReLU activation for hidden layer
    output_activations = np.dot(network.W2, hidden_activations) + network.b2

    # Softmax activation for output layer
    return neural_network_softmax(output_activations), hidden_activations


def neural_network_gradient_update(image, network: NN, gradient: NN_Grad, label):
    # Forward pass
    softmax_output, hidden_activations = neural_network_hypothesis(image, network)

    # Comute gradient
    dL_dsoftmax = np.copy(softmax_output)
    dL_dsoftmax[label] -= 1

    dL_dW2 = np.outer(dL_dsoftmax, hidden_activations)
    dL_db2 = dL_dsoftmax

    dL_dhidden = np.dot(network.W2.T, dL_dsoftmax)
    dL_dhidden[hidden_activations <= 0] = 0  # Gradient of ReLU activation

    dL_dW1 = np.outer(dL_dhidden, image)
    dL_db1 = dL_dhidden

    # Accumulate gradients
    gradient.W2_grad += dL_dW2
    gradient.b2_grad += dL_db2
    gradient.W1_grad += dL_dW1
    gradient.b1_grad += dL_db1
    epsilon = 1e-10
    # Calculate loss
    return -math.log(softmax_output[label] + epsilon)


def neural_network_training_step(dataset: dict, network: NN, learning_rate: float):
    total_loss = 0.0

    # Initialize gradient
    gradient = NN_Grad()

    # Calculate the gradient and the loss by looping through the training set
    for i in range(dataset["size"]):
        tmp = neural_network_gradient_update(
            dataset["images"][i], network, gradient, dataset["labels"][i]
        )
        total_loss += tmp

    for i in range(HIDDEN_SIZE):
        network.b1[i] -= learning_rate * gradient.b1_grad[i] / dataset["size"]
        for j in range(MNIST_IMAGE_SIZE):
            network.W1[i][j] -= learning_rate * gradient.W1_grad[i][j] / dataset["size"]

    for i in range(MNIST_LABELS):
        network.b2[i] -= learning_rate * gradient.b2_grad[i] / dataset["size"]
        for j in range(HIDDEN_SIZE):
            network.W2[i][j] -= learning_rate * gradient.W2_grad[i][j] / dataset["size"]

    return total_loss


def save_network(network: NN, filename: str):
    with open(filename, "wb") as f:
        pickle.dump(network, f)


In [ ]:
# MNIST Classification
# Chanyoung Ahn (https://github.com/cold-young)
# 24.06.13

# Command Example:
# $ python main_mnist.py

import numpy as np
import matplotlib.pyplot as plt

STEPS = 500
BATCH_SIZE = 400

# Dataset
# Downloaded from: http://yann.lecun.com/exdb/mnist/
data_sources = {
    "train_images": "data/train-images-idx3-ubyte",
    "train_labels": "data/train-labels-idx1-ubyte",
    "test_images": "data/t10k-images-idx3-ubyte",
    "test_labels": "data/t10k-labels-idx1-ubyte",
}


def calculate_accuracy(dataset: dict, network: NN):
    correct = 0

    for i in range(dataset["size"]):
        activations, _ = neural_network_hypothesis(dataset["images"][i], network)
        predict = np.argmax(activations)
        if predict == dataset["labels"][i]:
            correct += 1

    return correct / dataset["size"]


def main():
    np.random.seed(0)

    network = NN()
    loss, accuracy = float, float

    # # Read the dataset from the ./data directory
    train_dataset = mnist_get_dataset(
        data_sources["train_images"], data_sources["train_labels"]
    )
    test_dataset = mnist_get_dataset(
        data_sources["test_images"], data_sources["test_labels"]
    )

    np.random.seed(0)

    network.neural_network_random_weights()
    batches = train_dataset["size"] / BATCH_SIZE

    for i in range(STEPS):
        # Initialize a new batch
        batch = mnist_batch(train_dataset, BATCH_SIZE, i % batches)

        # Run one step of gradient descent and calculate the loss
        loss = neural_network_training_step(batch, network, 0.05)
        accuracy = calculate_accuracy(test_dataset, network)
        size = batch["size"]
        result = "Step: {0:3}  Average Loss: {1:2.3f} \t Accuracy: {2:3.3f}".format(
            i, loss / size, accuracy
        )
        print(result)
    save_network(network, "./serial_DNN_model.pkl")


if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        import traceback

        traceback.print_exc()


In [ ]:
%%writefile examples/mnist_file.py

import numpy as np
import os

current_directory = os.getcwd()


def mnist_get_dataset(images_path, labels_path):
    """
    Description:
    INPUT:
    OUTPUT:

    dataset.size
    dataset.label
    dataset.image
    """
    dataset = {}

    with open(os.path.join(current_directory, images_path), "rb") as mnist_file:
        dataset["images"] = np.frombuffer(
            mnist_file.read(), np.uint8, offset=16
        ).reshape(-1, 28 * 28)

    # Normalizate the data(images)
    dataset["images"] = dataset["images"] / 255.0

    with open(os.path.join(current_directory, labels_path), "rb") as mnist_file:
        dataset["labels"] = np.frombuffer(mnist_file.read(), np.uint8, offset=8)

    # (maybe) need one-hot encoding
    if len(dataset["images"]) != len(dataset["labels"]):
        print("Number of images does not match number of labels")
    else:
        dataset["size"] = len(dataset["images"])

    return dataset


# Fills the batch dataset with a subset of the parent dataset
def mnist_batch(dataset, size, number):
    """
    Description:
    INPUT:
    OUTPUT:
    """
    batch = {}

    start_offset = int(size * number)
    if start_offset >= dataset["size"]:
        return 0

    batch["images"] = dataset["images"][start_offset : start_offset + size]
    batch["labels"] = dataset["labels"][start_offset : start_offset + size]
    batch["size"] = size

    # print(f"start_offset:{start_offset}, batch:", batch["size"], ", dataset_size:", dataset["size"])
    if start_offset + batch["size"] > dataset["size"]:
        batch["size"] = dataset["size"] - start_offset
        print("batch_size:", batch["size"])

    return batch


In [ ]:
%%writefile examples/neural_network.py

import numpy as np
from mpi4py import MPI
import math
import pickle

MNIST_IMAGE_WIDTH = 28
MNIST_IMAGE_HEIGHT = 28
MNIST_IMAGE_SIZE = MNIST_IMAGE_WIDTH * MNIST_IMAGE_HEIGHT
MNIST_LABELS = 10
HIDDEN_SIZE = 128


class NN:
    def __init__(self):
        self.b1 = np.zeros(HIDDEN_SIZE, dtype=np.float32)
        self.W1 = np.zeros((HIDDEN_SIZE, MNIST_IMAGE_SIZE), dtype=np.float32)
        self.b2 = np.zeros(MNIST_LABELS, dtype=np.float32)
        self.W2 = np.zeros((MNIST_LABELS, HIDDEN_SIZE), dtype=np.float32)

    def neural_network_random_weights(self):
        self.b1 = np.random.rand(HIDDEN_SIZE).astype(np.float32)
        self.W1 = np.random.rand(HIDDEN_SIZE, MNIST_IMAGE_SIZE).astype(np.float32)
        self.b2 = np.random.rand(MNIST_LABELS).astype(np.float32)
        self.W2 = np.random.rand(MNIST_LABELS, HIDDEN_SIZE).astype(np.float32)


class NN_Grad:
    def __init__(self):
        self.b1_grad = np.zeros(HIDDEN_SIZE, dtype=np.float32)
        self.W1_grad = np.zeros((HIDDEN_SIZE, MNIST_IMAGE_SIZE), dtype=np.float32)
        self.b2_grad = np.zeros(MNIST_LABELS, dtype=np.float32)
        self.W2_grad = np.zeros((MNIST_LABELS, HIDDEN_SIZE), dtype=np.float32)


def neural_network_softmax(activations):
    e_activations = np.exp(activations - np.max(activations))
    return e_activations / np.sum(e_activations)


def neural_network_hypothesis(image, network):
    hidden_activations = np.maximum(
        0, np.dot(network.W1, image) + network.b1
    )  # ReLU activation for hidden layer
    output_activations = np.dot(network.W2, hidden_activations) + network.b2

    # Softmax activation for output layer
    return neural_network_softmax(output_activations), hidden_activations


def neural_network_gradient_update(image, network: NN, gradient: NN_Grad, label):
    # Forward pass
    softmax_output, hidden_activations = neural_network_hypothesis(image, network)

    # Comute gradient
    dL_dsoftmax = np.copy(softmax_output)
    dL_dsoftmax[label] -= 1

    dL_dW2 = np.outer(dL_dsoftmax, hidden_activations)
    dL_db2 = dL_dsoftmax

    dL_dhidden = np.dot(network.W2.T, dL_dsoftmax)
    dL_dhidden[hidden_activations <= 0] = 0  # Gradient of ReLU activation

    dL_dW1 = np.outer(dL_dhidden, image)
    dL_db1 = dL_dhidden

    # Accumulate gradients
    gradient.W2_grad += dL_dW2
    gradient.b2_grad += dL_db2
    gradient.W1_grad += dL_dW1
    gradient.b1_grad += dL_db1
    epsilon = 1e-10
    # Calculate loss
    return -math.log(softmax_output[label] + epsilon)


def neural_network_training_step(
    dataset: dict,
    network: NN,
    learning_rate: float,
    ista: int,
    iend: int,
    total_size: int,
):
    local_loss = np.array(0.0, dtype=np.float32)
    total_loss = np.array(0.0, dtype=np.float32)

    # Initialize gradient
    gradient = NN_Grad()

    # Calculate the gradient and the loss by looping through the training set
    for i in range(ista, iend + 1):
        local_loss += neural_network_gradient_update(
            dataset["images"][i], network, gradient, dataset["labels"][i]
        )

    comm = MPI.COMM_WORLD
    comm.Allreduce([local_loss, MPI.FLOAT], [total_loss, MPI.FLOAT], op=MPI.SUM)
    comm.Allreduce(MPI.IN_PLACE, gradient.W1_grad, op=MPI.SUM)
    comm.Allreduce(MPI.IN_PLACE, gradient.b1_grad, op=MPI.SUM)
    comm.Allreduce(MPI.IN_PLACE, gradient.W2_grad, op=MPI.SUM)
    comm.Allreduce(MPI.IN_PLACE, gradient.b2_grad, op=MPI.SUM)

    for i in range(HIDDEN_SIZE):
        network.b1[i] -= learning_rate * gradient.b1_grad[i] / dataset["size"]
        for j in range(MNIST_IMAGE_SIZE):
            network.W1[i][j] -= learning_rate * gradient.W1_grad[i][j] / dataset["size"]

    for i in range(MNIST_LABELS):
        network.b2[i] -= learning_rate * gradient.b2_grad[i] / dataset["size"]
        for j in range(HIDDEN_SIZE):
            network.W2[i][j] -= learning_rate * gradient.W2_grad[i][j] / dataset["size"]

    return total_loss


def save_network(network: NN, filename: str):
    with open(filename, "wb") as f:
        pickle.dump(network, f)


In [ ]:
%%writefile examples/main_mnist_parallel_DNN.py

# MNIST Classification
# Chanyoung Ahn (https://github.com/cold-young)
# 24.06.21

# Command Example:
# $ mpirun -np 4 python main_mnist.py

import numpy as np
import matplotlib.pyplot as plt
from mnist_file import mnist_get_dataset, mnist_batch
from neural_network import (
    NN,
    neural_network_training_step,
    neural_network_hypothesis,
    save_network,
)
from mpi4py import MPI

STEPS = 500
BATCH_SIZE = 400

# Dataset
# Downloaded from: http://yann.lecun.com/exdb/mnist/
data_sources = {
    "train_images": "../data/train-images-idx3-ubyte",
    "train_labels": "../data/train-labels-idx1-ubyte",
    "test_images": "../data/t10k-images-idx3-ubyte",
    "test_labels": "../data/t10k-labels-idx1-ubyte",
}


def calculate_accuracy(dataset: dict, network: NN, ista: int, iend: int):
    correct = np.array(0.0, dtype=np.float32)
    total_correct = np.array(0.0, dtype=np.float32)

    # for i in range(dataset["size"]):
    for i in range(ista, iend + 1):
        activations = neural_network_hypothesis(dataset["images"][i], network)
        predict = np.argmax(activations)
        if predict == dataset["labels"][i]:
            correct += 1.0

    comm = MPI.COMM_WORLD
    comm.Allreduce([correct, MPI.FLOAT], [total_correct, MPI.FLOAT], op=MPI.SUM)
    return total_correct / dataset["size"]


def para_range(N: int, nproc: int, myrank: int):
    iwork1 = N // nproc
    iwork2 = N % nproc
    ista = myrank * iwork1 + min(myrank, iwork2)
    iend = ista + iwork1 - 1
    if iwork2 > myrank:
        iend += 1
    return ista, iend


def main():
    np.random.seed(0)
    network = NN()
    loss, accuracy = float, float

    # MPI Initialize
    comm = MPI.COMM_WORLD
    myrank = comm.Get_rank()  # current_process
    nproc = comm.Get_size()  # np

    # # Read the dataset from the ./data directory
    train_dataset = mnist_get_dataset(
        data_sources["train_images"], data_sources["train_labels"]
    )
    test_dataset = mnist_get_dataset(
        data_sources["test_images"], data_sources["test_labels"]
    )

    batches = train_dataset["size"] / BATCH_SIZE

    ista1, iend1 = para_range(BATCH_SIZE, nproc, myrank)
    ista2, iend2 = para_range(test_dataset["size"], nproc, myrank)

    network.neural_network_random_weights()

    for i in range(STEPS):
        # Initialize a new batch
        batch = mnist_batch(train_dataset, BATCH_SIZE, i % batches)

        # Run one step of gradient descent and calculate the loss
        loss = neural_network_training_step(
            batch, network, 0.05, ista1, iend1, BATCH_SIZE
        )
        accuracy = calculate_accuracy(test_dataset, network, ista2, iend2)

        if myrank == 0:
            result = "Step: {0:3}  Average Loss: {1:2.3f} \t Accuracy: {2:3.3f}".format(
                i, loss / BATCH_SIZE, accuracy
            )
            print(result, flush=True)

    if myrank == 0:
        save_network(network, "./parallel_DNN_model.pkl")


if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        import traceback

        traceback.print_exc()


In [ ]:
! mpiexec -np 2 python examples/main_mnist_parallel_DNN.py